# Week 9 Lecture: Correlation & Regression Foundations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bradleyboehmke/uc-bana-4080/blob/main/notebooks/tuesday-your-turn/week-09-lecture.ipynb)

This week you build your first machine learning model. You'll measure how strongly two variables move together with correlation, turn that relationship into a prediction equation with linear regression, and then answer the question every manager will ask: *how good are these predictions?*

## How to use this notebook

This notebook accompanies the Week 9 Tuesday lecture. You can:

- **Follow along during class**: run each cell as the instructor discusses it
- **Work the Your Turn exercises after class**: they're practice for Thursday's lab
- **Reference after class**: use it alongside the textbook chapters

## This Week's Topics

- **Chapter 21: Correlation & Linear Regression**: measure relationships and fit a prediction line with scikit-learn
- **Chapter 22: Evaluating Regression Models**: score predictions with R² and RMSE, and use train/test splits to estimate performance on new data

## Setup

Run this cell first. It loads the libraries and the grocery chain data we use throughout the lecture: 20 weeks of advertising spend and weekly sales.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error

# 20 weeks of advertising spend vs. weekly sales for a grocery chain
data = pd.DataFrame({
    "ad_spend": [400, 500, 600, 700, 800, 900, 1000, 1100, 1200, 1300,
                 1400, 1500, 1600, 1700, 1800, 1900, 2000, 2100, 2200, 2300],
    "weekly_sales": [4200, 4400, 4100, 4800, 5600, 5200, 4900, 5500, 5300, 5900,
                     5700, 6300, 6900, 6200, 5800, 6600, 7100, 6800, 7300, 7800]
})

data.head()

---

## 1. Correlation: Do Two Variables Move Together?

Before building a model, look at the relationship. Each point below is one week.

**Before running the next cell:** write down your guess for the correlation between ad spend and weekly sales (somewhere between -1 and +1).

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(data["ad_spend"], data["weekly_sales"], alpha=0.7, s=60)
plt.xlabel("Advertising Spend ($)")
plt.ylabel("Weekly Sales")
plt.title("Ad Spend vs. Weekly Sales")
plt.grid(True, alpha=0.3)
plt.show()

`.corr()` computes the correlation between two columns. Call it on the whole DataFrame to get a correlation matrix for every pair of numeric columns.

In [ ]:
# Correlation between two columns
correlation = data["ad_spend"].corr(data["weekly_sales"])
print(f"Correlation coefficient: {correlation:.3f}")

# Correlation matrix for every pair of columns
data.corr()

> **Correlation ≠ causation.** A strong correlation tells you two variables move together, not *why*.

---

## 2. Linear Regression: From Relationship to Prediction

Linear regression finds the line $y = \beta_0 + \beta_1 x$ that best fits the data:

- $\beta_0$ (**intercept**): predicted sales when ad spend is $0
- $\beta_1$ (**slope**): how much sales change for each extra $1 of ad spend

scikit-learn expects the features `X` as a DataFrame (note the double brackets) and the target `y` as a single column.

In [ ]:
X = data[["ad_spend"]]   # features: a DataFrame, so double brackets
y = data["weekly_sales"]   # target: a single column

model = LinearRegression()
model.fit(X, y)

print(f"Intercept (β₀): {model.intercept_:,.0f}")
print(f"Slope (β₁):     {model.coef_[0]:.2f}")

**Interpretation:** for every additional $1 of advertising, the model expects weekly sales to rise by about $1.68.

Every prediction the model makes falls on the fitted line:

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(data["ad_spend"], data["weekly_sales"], alpha=0.7, s=60, label="Actual data")
plt.plot(data["ad_spend"], model.predict(X), color="red", linewidth=3, label="Regression line")
plt.xlabel("Advertising Spend ($)")
plt.ylabel("Weekly Sales")
plt.title("Linear Regression: Ad Spend vs. Weekly Sales")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

To forecast new scenarios, pass `.predict()` a DataFrame with the **same column name** the model was trained on.

In [ ]:
scenarios = pd.DataFrame({"ad_spend": [1500, 2500, 3000]})

scenarios.assign(predicted_sales=model.predict(scenarios).round(0))

### 🎯 Your Turn: Make Your Own Forecasts

1. Use `model` to predict weekly sales for ad budgets of **$1,000** and **$1,800**.
2. Check one of your predictions **by hand** using the intercept and slope: `intercept + slope × ad_spend`. Does it match?

In [ ]:
# Your code here

---

## 3. Evaluating Model Performance

To evaluate a model, compare its **predictions** with what **actually happened**, then summarize the misses with a metric.

| Metric | Asks | Units | Better |
|--------|------|-------|--------|
| **RMSE** | How far off are my predictions, typically? | Same as the target ($) | Lower |
| **R²** | How much better is my model than guessing the average? | None (usually 0 to 1) | Higher |

Both functions take the **actual values first**, then the **predictions**.

In [ ]:
fitted_sales = model.predict(X)

r2 = r2_score(y, fitted_sales)
rmse = root_mean_squared_error(y, fitted_sales)

print(f"R²:   {r2:.2f}")
print(f"RMSE: ${rmse:,.0f}")

**But there's a catch:** we just scored the model on the same 20 weeks it learned from. That's like getting the exam questions the night before. What we really want to know is how the model performs on data it has **never seen**.

### Train/Test Split

Hold some data back:

- **Training set** (70–80%): the model learns the pattern here
- **Test set** (20–30%): locked away, then used once to score the model

The test score is our best preview of how the model will perform on future data.

In [ ]:
# Hold out 30% of the weeks as a test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=11
)

print(f"Training weeks: {len(X_train)}")
print(f"Test weeks:     {len(X_test)}")

In [ ]:
# Learn ONLY from the training weeks
model = LinearRegression()
model.fit(X_train, y_train)

# Score the model on both sets
train_preds = model.predict(X_train)
test_preds = model.predict(X_test)

print(f"Training → R²: {r2_score(y_train, train_preds):.2f} | RMSE: ${root_mean_squared_error(y_train, train_preds):,.0f}")
print(f"Test     → R²: {r2_score(y_test, test_preds):.2f} | RMSE: ${root_mean_squared_error(y_test, test_preds):,.0f}")

**Reading the gap:**

- **Train ≈ Test**: the model generalizes; trust the test score
- **Train great, Test much worse**: the model memorized the training data (*overfitting*)
- **Train poor, Test poor**: the model is too simple or missing key information (*underfitting*)

A test score slightly worse than the training score is normal.

### 🎯 Your Turn: Does the Split Matter?

1. Copy the split-train-score code from the two cells above and rerun it with a **different `random_state`** (pick any number). How much does the test RMSE change?
2. Now try **`test_size=0.2`**. How many test weeks do you get, and what happens to the test RMSE?
3. In a sentence or two: why does the test score change so much with this dataset?

In [ ]:
# Your code here

*Your answer to question 3:*

---

## 4. Putting It All Together

### 🎯 Your Turn: Predict Home Prices in Ames

You're helping a local realtor estimate home sale prices. Use the Ames housing data from Week 4 to build and evaluate a model that uses **above-ground living area** (`Gr Liv Area`, in square feet) to predict **sale price** (`SalePrice`).

Follow the four steps of the workflow:

1. **Load** the data (code provided below) and make a scatter plot of `Gr Liv Area` vs. `SalePrice`. What's the correlation?
2. **Split** the data into training and test sets (use `test_size=0.2` and `random_state=42`).
3. **Train** a `LinearRegression` model on the training set. Interpret the slope: what does one extra square foot add to the predicted price?
4. **Evaluate** the model on both sets with R² and RMSE. Does the model generalize?

Then answer: **how would you explain the test RMSE to the realtor in one plain-English sentence?**

In [ ]:
url = "https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/data/ames_raw.csv"
ames = pd.read_csv(url)

ames[["Gr Liv Area", "SalePrice"]].head()

In [ ]:
# Step 1: Explore the relationship

# Step 2: Split the data

# Step 3: Train the model

# Step 4: Evaluate on training and test sets

*Your explanation for the realtor:*

---

## 🧾 What You Learned

| Code | What it does |
|------|--------------|
| `df["a"].corr(df["b"])` | Correlation between two columns (-1 to +1) |
| `df.corr()` | Correlation matrix for every pair of numeric columns |
| `X = df[["feature"]]`, `y = df["target"]` | Set up features (DataFrame) and target (column) |
| `model = LinearRegression().fit(X, y)` | Fit a linear regression model |
| `model.intercept_`, `model.coef_` | The fitted intercept and slope(s) |
| `model.predict(new_X)` | Predict for new data (same column names as training) |
| `r2_score(y_true, y_pred)` | Share of variation explained (higher is better) |
| `root_mean_squared_error(y_true, y_pred)` | Typical size of a miss, in the target's units (lower is better) |
| `train_test_split(X, y, test_size=0.3, random_state=11)` | Hold out a test set to estimate performance on new data |

## What's Next

- **Thursday Lab:** hands-on regression modeling and evaluation practice. This week's lab is also your homework.
- **Chapters to review:** [Chapter 21](../../book/21-correlation-regression.html) and [Chapter 22](../../book/22-regression-evaluation.html) for deeper coverage, including multiple regression, categorical variables, and MAE/MAPE
- **Cheat sheet:** [Module 9 Cheat Sheet](../../book/module-09-cheatsheet.html) for a quick reference on all of today's concepts